In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import json
import time
import pandas as pd

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10
RANDOM_STATE = 1618 # same random state as default for MATAVE topic model

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def nmf_analysis(texts, dataset_name):
    # Prepare components for evaluation.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    # Vectorize texts for NMF.
    vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
    )
    tfidf = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()

    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    for k in K_RANGE:
        start = time.time()

        nmf_model = NMF(
            n_components=k,
            random_state=RANDOM_STATE
        )
        nmf_model.fit(tfidf)

        nmf_topics = [
            [feature_names[i] for i in topic.argsort()[:-TOP_N - 1:-1]]
            for topic in nmf_model.components_
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'NMF (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(nmf_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(nmf_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(nmf_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(nmf_topics)

    return metric_results

In [6]:
all_results = []
all_texts = []
for key in nurse_notes:
    all_results.append(pd.DataFrame(nmf_analysis(nurse_notes[key], key)))
    all_texts.extend(nurse_notes[key])
combined_df = pd.concat(all_results)

/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maxim

In [7]:
def get_top_topics(temp_df):
    metrics = ['Coherence', 'Diversity', 'Redundancy']

    for metric in metrics:
        print(f"-------{metric}-------")
        temp_top_row = temp_df.loc[temp_df[metric].idxmax()]['Top Topic Words']
        print(f"Number of Topics: {len(temp_top_row)}")
        for topic in temp_top_row:
            # print(' '.join(topic))
            print(topic)
        print("\n")

In [8]:
for patient in list(nurse_notes.keys()):
    print("----------------------------")
    print(patient)
    print("----------------------------")
    get_top_topics(combined_df[combined_df['Dataset Name'] == patient])

----------------------------
P1
----------------------------
-------Coherence-------
Number of Topics: 6
['complaint', 'rollator', 'mobilise', 'voice', 'nil', 'good', 'form', 'appear', 'need', 'chart']
['ongoing', 'asleep', 'toilette', 'self', 'check', 'comfortable', 'peaceful', 'resident', 'change', 'toilete']
['check', 'safety', 'sleep', 'settle', 'night', 'continue', 'concern', 'bed', 'comfortable', 'med']
['plan', 'morning', 'staff', 'care', 'report', 'adls', 'breakfast', 'intake', 'assist', 'night']
['restaurant', 'attend', 'mobile', 'need', 'usual', 'independent', 'form', 'chart', 'meal', 'med']
['mobilizing', 'relaxed', 'walker', 'med', 'chart', 'appear', 'staff', 'content', 'assisted', 'assist']


-------Diversity-------
Number of Topics: 4
['need', 'form', 'attend', 'chart', 'good', 'complaint', 'nil', 'rollator', 'appear', 'mobilise']
['ongoing', 'asleep', 'toilette', 'self', 'check', 'comfortable', 'peaceful', 'resident', 'change', 'toilete']
['check', 'safety', 'sleep', 'ni

In [10]:
all_texts_df = pd.DataFrame(nmf_analysis(all_texts, "All Texts"))

/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [12]:
get_top_topics(all_texts_df)

-------Coherence-------
Number of Topics: 4
['sleep', 'settle', 'medication', 'night', 'nocte', 'continue', 'bed', 'overnight', 'administer', 'early']
['good', 'form', 'chart', 'appear', 'med', 'nil', 'care', 'assist', 'attend', 'resident']
['asleep', 'check', 'ongoing', 'comfortable', 'care', 'need', 'skin', 'continue', 'peaceful', 'assist']
['night', 'assisted', 'maintain', 'charted', 'compliant', 'meds', 'adls', 'safety', 'check', 'settle']


-------Diversity-------
Number of Topics: 4
['sleep', 'settle', 'medication', 'night', 'nocte', 'continue', 'bed', 'overnight', 'administer', 'early']
['good', 'form', 'chart', 'appear', 'med', 'nil', 'care', 'assist', 'attend', 'resident']
['asleep', 'check', 'ongoing', 'comfortable', 'care', 'need', 'skin', 'continue', 'peaceful', 'assist']
['night', 'assisted', 'maintain', 'charted', 'compliant', 'meds', 'adls', 'safety', 'check', 'settle']


-------Redundancy-------
Number of Topics: 18
['check', 'comfortable', 'safety', 'asleep', 'med', 'c